# F1 Winner Prediction - Notebook 03: Ablation Study

## Por que NO usamos fine-tuning separado?

Demuestra que fine-tuning en solo 37 secuencias (2023-2024) causa overfitting severo.
Entrenar todo junto (2014-2024, 192 seqs) es la estrategia correcta para datasets pequenos.

In [ ]:
# @title 1. Clone Repo & Setup
!git clone https://github.com/USERNAME/f1_transformer.git 2>/dev/null || echo 'Repo already cloned'
%cd f1_transformer

from google.colab import drive
drive.mount('/content/drive')

import os; os.environ['COLAB'] = '1'

import sys; from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import torch, torch.nn as nn
import numpy as np, pickle, matplotlib.pyplot as plt, time
from tqdm.notebook import tqdm

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
print(f'Device: {"cuda" if torch.cuda.is_available() else "cpu"}')

In [ ]:
# @title 2. Load Tiny FT Dataset & Pre-trained Model
from src.model.transformer_model import F1WinnerTransformer
from src.training.trainer import Trainer

PROCESSED = Path('/content/drive/MyDrive/f1_transformer/data/processed')
FINETUNED_DIR = Path('/content/drive/MyDrive/f1_transformer/models/finetuned')
FINETUNED_DIR.mkdir(parents=True, exist_ok=True)

ft_train_data = torch.load(PROCESSED / 'features_finetune.pt', weights_only=False)
ft_val_data = torch.load(PROCESSED / 'features_finetune_val.pt', weights_only=False)

with open(PROCESSED / 'metadata.pkl', 'rb') as f:
    metadata = pickle.load(f)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = torch.cuda.is_available()

print(f'FT train: {len(ft_train_data["winners"])} sequences')
print(f'FT val:   {len(ft_val_data["winners"])} sequences')
print(f'')
print(f'WARNING: Solo {len(ft_train_data["winners"])} secuencias para fine-tuning!')
print(f'Un transformer de 2.28M params necesita mas datos para generalizar.')

In [ ]:
# @title 3. Run Fine-Tuning Experiment (Observe Overfitting)

# Create model
model = F1WinnerTransformer(
    d_model=192, n_heads=6, n_encoder_layers=3, n_cross_attn_layers=2,
    d_ff=768, dropout=0.15,
    context_window=metadata['context_window'],
    num_drivers=metadata['num_drivers'],
    num_constructors=metadata['num_constructors'],
    num_circuits=metadata['num_circuits'],
    d_candidate_raw=metadata['d_candidate_raw'],
    d_context_raw=metadata['d_context_raw'],
)

# Load combined model as starting point (simulates pre-training on 2014-2022)
COMBINED_MODEL = Path('/content/drive/MyDrive/f1_transformer/models/final/best.pt')
ckpt = torch.load(COMBINED_MODEL, map_location='cpu', weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
print(f'Loaded base model (val_acc={ckpt.get("best_val_acc",0):.4f})')

# Freeze first 2 encoder layers (standard FT practice)
for i, layer in enumerate(model.race_encoder.layers):
    if i < 2:
        for p in layer.parameters():
            p.requires_grad = False
        print(f'Frozen encoder layer {i}')

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable: {trainable:,} / {sum(p.numel() for p in model.parameters()):,}')

# FT with lower learning rate
ft_trainer = Trainer(
    model=model, train_data=ft_train_data, val_data=ft_val_data,
    device=DEVICE, batch_size=8, learning_rate=5e-5,
    weight_decay=1e-4, epochs=25, patience=10,
    use_amp=USE_AMP, checkpoint_dir=FINETUNED_DIR,
)

print('\nStarting fine-tuning...')
ft_history = ft_trainer.train(early_stopping=True)

In [ ]:
# @title 4. Visualize Overfitting
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(ft_history['train_loss'], label='Train Loss (37 seqs)', linewidth=2, color='blue')
axes[0].plot(ft_history['val_loss'], label='Val Loss (9 seqs)', linewidth=2, color='red')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Fine-Tuning: Train Loss Drops, Val Loss DIVERGES')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(ft_history['val_acc'], label='FT Val Accuracy', linewidth=2, color='red')
axes[1].axhline(y=0.545, color='blue', ls='--', alpha=0.7, label='Combined Training (54.5%)')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].set_title('Fine-Tuning Accuracy: Degrades Significantly')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'\nFT best val acc: {ft_trainer.best_val_acc:.3f}')
print(f'Combined training val acc: 0.545')
print(f'FT DEGRADATION: {(0.545-ft_trainer.best_val_acc)*100:.1f} percentage points worse!')

In [ ]:
# @title 5. Conclusions
print('='*65)
print('ABLATION RESULTS')
print('='*65)
print(f'')
print(f'  Approach                  | Train Seqs | Val Acc  | 2025 Acc')
print(f'  --------------------------+-----------+---------+---------')
print(f'  Pre-train (2014-2022)     | 153       | ~61.5%  | N/A')
print(f'  + Fine-tune (2023-2024)   | 37        | ~22.2%  | N/A     <-- OVERFITS')
print(f'  Combined (2014-2024)      | 192       | 54.5%   | 41.7%   <-- BEST')
print(f'')
print(f'Why FT fails:')
print(f'  1. 37 seqs for 2.28M params = severe overfitting')
print(f'  2. Val loss DIVERGES while train loss drops')
print(f'  3. Model memorizes 2023-2024 instead of generalizing')
print(f'')
print(f'KEY INSIGHT: For F1 datasets (< 300 sequences),')
print(f'COMBINED training on all available data outperforms')
print(f'the pre-train + fine-tune paradigm.')
print(f'')
print('\nReady for Notebook 04: 2025 Predictions!')